In [5]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import sqlite3

In [12]:
df = pd.read_csv("Sample - Superstore.csv", encoding="latin1")

In [13]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

In [15]:
df.shape

(9994, 21)

### Converting column name to lower case


In [18]:
df.columns=df.columns.str.replace(' ','_').str.lower()

In [19]:
df.columns

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
       'postal_code', 'region', 'product_id', 'category', 'sub-category',
       'product_name', 'sales', 'quantity', 'discount', 'profit'],
      dtype='object')

In [17]:
df.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub-category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


### Creating a temporary SQLite database in RAM.

In [21]:
conn = sqlite3.connect(":memory:")

### Store the DataFrame as a SQL table

In [22]:
df.to_sql("superstore", conn, if_exists="replace", index=False)

9994

### 1. Total Sales and Profit grouped by Region.

In [23]:
query = """
SELECT
    region,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit
FROM superstore
GROUP BY region
ORDER BY total_sales DESC;
"""

result = pd.read_sql_query(query, conn)
result

,region,total_sales,total_profit
0,West,725457.8245,108418.4489
1,East,678781.2400,91522.7800
2,Central,501239.8908,39706.3625
3,South,391721.9050,46749.4303


### 2. Total Profit grouped by Category.

In [25]:
query= """
SELECT 
    category, SUM(profit) as total_profit
    FROM superstore
    GROUP BY category
    ORDER BY total_profit DESC;
    """
result = pd.read_sql_query(query, conn)
result

,category,total_profit
0,Technology,145454.9481
1,Office Supplies,122490.8008
2,Furniture,18451.2728


### 3. Sub-categories where average Profit is negative (GROUP BY + HAVING).

In [27]:
query = """
SELECT
    `sub-category` AS sub_category,
    AVG(profit) AS avg_profit
FROM superstore
GROUP BY `sub-category`
HAVING AVG(profit) < 0
ORDER BY avg_profit;
"""

result = pd.read_sql_query(query, conn)
result

,sub_category,avg_profit
0,Tables,-55.565771
1,Bookcases,-15.230509
2,Supplies,-6.258418


### 4. Average Discount grouped by Category.

In [28]:
query = """
SELECT
    category,
    AVG(discount) AS avg_discount
FROM superstore
GROUP BY category
ORDER BY avg_discount;
"""

result = pd.read_sql_query(query, conn)
result

,category,avg_discount
0,Technology,0.132323
1,Office Supplies,0.157285
2,Furniture,0.173923


### 5. List all orders with Profit below the overall average Profit using a subquery.

In [30]:
query = """
SELECT
    order_id,
    category,
    `sub-category` AS sub_category,
    sales,
    profit
FROM superstore
WHERE profit < (
    SELECT AVG(profit)
    FROM superstore
)
ORDER BY profit;
"""

result = pd.read_sql_query(query, conn)
result

,order_id,category,sub_category,sales,profit
0,CA-2016-108196,Technology,Machines,4499.985,-6599.9780
1,US-2017-168116,Technology,Machines,7999.980,-3839.9904
2,CA-2014-169019,Office Supplies,Binders,2177.584,-3701.8928
3,CA-2017-134845,Technology,Machines,2549.985,-3399.9800
4,US-2017-122714,Office Supplies,Binders,1889.990,-2929.4845
...,...,...,...,...,...
7438,CA-2014-144071,Furniture,Furnishings,79.440,28.5984
7439,CA-2016-125206,Office Supplies,Storage,114.460,28.6150
7440,CA-2017-132682,Technology,Phones,381.576,28.6182
7441,US-2017-116897,Office Supplies,Storage,477.150,28.6290
